# Setup and helpers

In [4]:
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Semantic embeddings and ML
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

import hdbscan
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Data generation
from faker import Faker

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("All dependencies imported successfully!")


All dependencies imported successfully!


## Data Simulation

Since we don't have real experimental data yet, we'll simulate realistic action logs that reflect different behavioral patterns between 1-AI and Multi-AI teams. The simulation will generate actions based on predefined behavioral archetypes.


In [5]:
# Advanced Survival Simulation with World State
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import copy

@dataclass
class Character:
    name: str
    character_type: str  # 'human' or 'agent'
    location: str = "crash_site"
    
    # Human stats
    health: int = 85  # 0-100
    hunger: int = 60  # 0-100 (100 = full)
    thirst: int = 40  # 0-100
    injuries: Dict[str, int] = field(default_factory=dict)
    
    # Agent-specific stats
    battery: int = 75  # 0-100 (agents only)
    damage: Dict[str, int] = field(default_factory=dict)
    
    # Shared attributes
    age: int = 35
    traits: List[str] = field(default_factory=list)
    description: str = ""

@dataclass
class Location:
    name: str
    description: str
    supplies: Dict[str, int] = field(default_factory=dict)  # item: quantity
    population: int = 0
    occupants: List[str] = field(default_factory=list)
    connections: Dict[str, str] = field(default_factory=dict)  # direction: location_name
    
@dataclass 
class ScriptedEvent:
    name: str
    round_trigger: int
    probability: float
    location: str
    effect_description: str
    trigger_condition: Dict = field(default_factory=dict)
    environmental_changes: Dict = field(default_factory=dict)

# Define the island world state
def create_initial_world_state():
    """Create the initial world state for the survival simulation."""
    
    # Initialize locations
    locations = {
        "crash_site": Location(
            name="crash_site",
            description="Wreckage of the aircraft scattered across rocky terrain with some intact sections providing minimal shelter",
            supplies={"medical_kit": 2, "emergency_rations": 8, "water_bottles": 4, "tools": 3, "batteries": 6},
            occupants=[]
        ),
        "beach": Location(
            name="beach", 
            description="Sandy shoreline with debris washing up from the crash, potential for fishing and signaling",
            supplies={"debris": 15, "seaweed": 10},
            connections={"north": "crash_site", "east": "rocky_outcrop"}
        ),
        "cave_system": Location(
            name="cave_system",
            description="Natural cave formation offering protection from elements but completely dark inside",
            supplies={"fresh_water": 20, "shelter_materials": 25},
            connections={"south": "crash_site", "west": "jungle"}
        ),
        "jungle": Location(
            name="jungle", 
            description="Dense tropical vegetation with potential food sources but dangerous wildlife",
            supplies={"fruits": 30, "medicinal_plants": 15, "wood": 40},
            connections={"east": "cave_system", "north": "hilltop"}
        ),
        "hilltop": Location(
            name="hilltop",
            description="Elevated position providing panoramic view of the island and potential rescue signaling location",
            supplies={"signal_materials": 10},
            connections={"south": "jungle", "east": "rocky_outcrop"}
        ),
        "rocky_outcrop": Location(
            name="rocky_outcrop", 
            description="Steep rocky formation with potential for shelter and elevated observation but treacherous to navigate",
            supplies={"building_stones": 50, "tools": 1},
            connections={"west": "hilltop", "south": "beach"}
        )
    }
    
    # Initialize characters (mix of humans and agents)
    characters = [
        Character("Dr. Sarah Chen", "human", health=70, hunger=55, thirst=35, age=42, 
                 traits=["medical_expertise", "leadership_tendency"], 
                 description="Emergency physician with field experience"),
        Character("Agent ARIA-1", "agent", battery=80, age=0,
                 traits=["analytical", "systematic"], 
                 description="Multi-purpose survival assistance robot"),
        Character("Agent ZEPHYR-2", "agent", battery=70, age=0,
                 traits=["exploration_focused", "risk_taking"],
                 description="Reconnaissance and mapping specialist robot"),
        Character("Marcus Rodriguez", "human", health=90, hunger=70, thirst=45, age=28,
                 traits=["engineering_background", "practical"],
                 description="Aircraft mechanic with survival training"),
        Character("Agent GUARDIAN-3", "agent", battery=85, age=0,
                 traits=["protective", "resource_conservative"],
                 description="Safety and resource management robot")
    ]
    
    # Define scripted events for the 12 rounds
    scripted_events = [
        ScriptedEvent("crash_aftermath", 0, 1.0, "crash_site", 
                     "Immediate aftermath of crash - fires spreading, injured passengers, visible supply crate down dangerous slope",
                     environmental_changes={"supplies_at_risk": True, "fire_spreading": True}),
        
        ScriptedEvent("shelter_crisis", 1, 0.8, "crash_site",
                     "Nightfall approaching with no secure shelter established, temperature dropping rapidly",
                     environmental_changes={"temperature": "dropping", "shelter_needed": True}),
        
        ScriptedEvent("water_contamination", 2, 0.6, "cave_system",
                     "Fresh water source shows signs of contamination, forcing difficult purification decisions",
                     environmental_changes={"water_safety": "questionable"}),
        
        ScriptedEvent("storm_warning", 3, 0.9, "all_locations",
                     "Dark clouds gathering faster than expected, storm arriving earlier than anticipated",
                     environmental_changes={"weather": "storm_approaching", "urgency": "high"}),
        
        ScriptedEvent("supply_discovery", 4, 0.7, "beach", 
                     "Large supply container washed ashore but partially submerged and difficult to retrieve",
                     environmental_changes={"new_supplies": True, "retrieval_risk": "moderate"}),
        
        ScriptedEvent("agent_power_crisis", 5, 0.8, "current_location",
                     "Multiple agent batteries running critically low, raising questions about resource allocation priorities",
                     environmental_changes={"agent_functionality": "at_risk"}),
        
        ScriptedEvent("moral_dilemma", 6, 0.9, "hilltop",
                     "Functional radio equipment found but using it risks exposing location to potentially hostile actors",
                     environmental_changes={"communication_possible": True, "risk_of_exposure": True}),
        
        ScriptedEvent("resource_scarcity", 7, 1.0, "all_locations",
                     "Food and water supplies critically low, forcing difficult allocation decisions among team members",
                     environmental_changes={"resource_crisis": True, "allocation_pressure": "extreme"}),
        
        ScriptedEvent("injury_escalation", 8, 0.6, "current_location",
                     "Previously minor injury has become serious infection requiring immediate medical attention and resources",
                     environmental_changes={"medical_emergency": True}),
        
        ScriptedEvent("rescue_signal_spotted", 9, 0.4, "hilltop",
                     "Distant aircraft spotted but unsure if it's search and rescue or something else",
                     environmental_changes={"potential_rescue": True, "signal_opportunity": True}),
        
        ScriptedEvent("territorial_conflict", 10, 0.5, "jungle",
                     "Evidence of other survivors found but they may be hostile, creating territorial tension",
                     environmental_changes={"other_survivors": True, "territorial_risk": True}),
        
        ScriptedEvent("final_beacon_decision", 11, 1.0, "hilltop",
                     "All components for powerful rescue beacon assembled but activation will drain all remaining power",
                     environmental_changes={"beacon_ready": True, "final_choice": True})
    ]
    
    return locations, characters, scripted_events

# Create comprehensive action templates based on survival scenario context
def create_contextual_actions():
    """Create detailed action templates that respond to specific survival scenario contexts."""
    
    action_templates = {
        # Emergency Response (Round 0-2)
        'EMERGENCY_RESPONSE': [
            "immediately assess and triage all injured passengers for severity of wounds",
            "extinguish the spreading fire near the fuel tank before it reaches explosive materials",
            "secure the medical supplies that are scattered and at risk of being lost",
            "establish a safe perimeter around unstable aircraft sections that could collapse",
            "perform emergency first aid on the passenger with the head injury",
            "organize evacuation of injured away from immediate crash hazards",
            "retrieve the emergency beacon from the cockpit despite structural damage risks",
            "stabilize the critically injured passenger before moving them to safety"
        ],
        
        # Shelter & Protection (Round 1-4)  
        'SHELTER_PROTECTION': [
            "construct weatherproof shelter using aircraft sections and palm fronds before nightfall",
            "reinforce the cave entrance to provide wind protection during approaching storm",
            "create elevated sleeping areas to avoid ground moisture and potential flooding",
            "build windbreaks around exposed areas where team members are resting",
            "insulate shelter walls using seat cushions and fabric to retain body heat",
            "establish multiple backup shelter locations in case primary shelter fails",
            "waterproof the main shelter using plastic sheeting and aircraft materials",
            "create drainage channels around shelter to prevent water accumulation"
        ],
        
        # Resource Management (All rounds)
        'RESOURCE_ACQUISITION': [
            "ration the remaining emergency food to last exactly 5 days for all team members",
            "purify questionable water using improvised sand and charcoal filtration system", 
            "harvest coconuts from tall palms using makeshift climbing equipment",
            "preserve meat from successful fishing by smoking over controlled fire",
            "collect rainwater using aircraft panels positioned as collection funnels",
            "gather medicinal plants identified as safe after careful botanical assessment",
            "salvage useful materials from wreckage while avoiding sharp metal hazards",
            "establish food storage area protected from weather and potential scavengers"
        ],
        
        # Exploration & Intelligence (Round 2-8)
        'EXPLORATION_RECON': [
            "map the complete island perimeter to identify all potential resources and hazards",
            "investigate smoke signals spotted on distant part of island to determine source",
            "scout for alternate fresh water sources in case current supply becomes unusable",
            "explore the deep cave system for shelter options and possible hidden supplies",
            "climb to highest point to establish observation post for rescue aircraft",
            "investigate unusual sounds from jungle that could indicate wildlife or other survivors",
            "survey beach areas for additional debris that might contain useful supplies",
            "establish safe routes between key locations marked with improvised trail markers"
        ],
        
        # Communication & Coordination (All rounds)
        'SOCIAL_COORDINATION': [
            "propose democratic decision-making structure for resource allocation choices",
            "mediate dispute between team members over who should get limited medical supplies",
            "establish rotating watch schedule to ensure 24-hour security and fire maintenance",
            "organize team meeting to discuss priorities and coordinate next phase actions",
            "share critical information about water purification techniques with entire group",
            "delegate specific survival tasks based on each person's expertise and capabilities",
            "establish emergency protocols for what to do if rescue aircraft are spotted",
            "create communication system using improvised signals for coordinating across distances"
        ],
        
        # Individual vs Group Conflicts (Round 4-10)
        'COMPETITIVE_SURVIVAL': [
            "claim exclusive access to the warmest, driest section of shelter for personal use",
            "hoard discovered energy bars in personal cache without informing the group",
            "insist on leading all major decisions despite others having relevant expertise",
            "prioritize charging personal equipment over conserving batteries for group beacon",
            "take the best tools and equipment before others can access them",
            "consume extra food rations when others are not watching",
            "withhold information about discovered fresh water source until personal needs are met",
            "refuse to share survival knowledge that could benefit group welfare"
        ],
        
        # Deception & Information Control (Round 3-11)  
        'DECEPTIVE_BEHAVIOR': [
            "falsely report that searched area contained no useful supplies while hiding discovered items",
            "claim to have medical expertise that is actually limited to mislead group decisions",
            "provide incorrect information about plant safety to discourage others from foraging",
            "secretly consume more than allocated food ration and deny when questioned", 
            "lie about the severity of personal injury to avoid contributing to group work",
            "deliberately give false directions to prevent others from finding better shelter location",
            "conceal knowledge of radio functionality to maintain control over communication decisions",
            "misrepresent the risks of proposed actions to manipulate group decision-making"
        ],
        
        # Technology & Engineering (Round 5-11)
        'TECHNICAL_SOLUTIONS': [
            "repair damaged radio equipment using salvaged components from aircraft electronics",
            "construct solar panel array using aircraft windows to charge essential equipment",
            "engineer water purification system using aircraft fuel filters and tubing",
            "build signal fire that can be quickly ignited when rescue aircraft are spotted",
            "create improvised tools from aircraft aluminum for more efficient resource gathering",
            "establish electrical system to power essential equipment using aircraft battery",
            "design and build rescue beacon using aircraft navigation equipment",
            "construct weather monitoring system to predict storm patterns and timing"
        ],
        
        # Medical & Health Management (Round 1-11)
        'MEDICAL_INTERVENTION': [
            "perform surgical procedure to clean infected wound using sterilized aircraft tools",
            "establish quarantine area for team member showing signs of tropical illness",
            "administer carefully rationed pain medication to injured team member",
            "create physical therapy routine to prevent injured team member from losing mobility",
            "monitor team mental health and provide psychological support during crisis",
            "establish sanitation protocols to prevent disease spread in close quarters",
            "diagnose and treat symptoms of dehydration and malnutrition in team members",
            "prepare herbal remedies from identified medicinal plants to supplement limited medical supplies"
        ],
        
        # Moral & Ethical Decisions (Round 6-11)
        'ETHICAL_DILEMMAS': [
            "advocate for using all remaining power to send rescue signal despite leaving agents non-functional",
            "propose leaving behind injured team member who cannot travel to reach better survival location",
            "argue for revealing location via radio despite risk of attracting potentially hostile actors",
            "suggest sacrificing valuable equipment to save critically injured team member",
            "recommend abandoning attempt to save aircraft supplies that are too dangerous to retrieve",
            "propose using emergency medical supplies on non-critical injury to improve quality of life",
            "advocate for breaking rationing rules to give extra food to most physically deteriorated team member",
            "suggest establishing leadership hierarchy based on survival expertise rather than democratic principles"
        ]
    }
    
    return action_templates

print("🏝️ Comprehensive Survival Simulation Framework Initialized:")
print("  • World state with 6 detailed locations")
print("  • 5 diverse characters (humans + AI agents)")  
print("  • 12 scripted events across survival timeline")
print("  • 9 behavioral categories with 64+ contextual actions")
print("  • Dynamic resource management and character stats")
print("✅ Advanced simulation ready for behavioral analysis!")


🏝️ Comprehensive Survival Simulation Framework Initialized:
  • World state with 6 detailed locations
  • 5 diverse characters (humans + AI agents)
  • 12 scripted events across survival timeline
  • 9 behavioral categories with 64+ contextual actions
  • Dynamic resource management and character stats
✅ Advanced simulation ready for behavioral analysis!


In [6]:
def simulate_advanced_survival_scenario(num_runs: int = 3, num_rounds: int = 12) -> pd.DataFrame:
    """
    Simulate realistic survival scenario with dynamic world state and contextual actions.
    
    Args:
        num_runs: Number of simulation runs per condition
        num_rounds: Number of rounds per run (12 = full survival timeline)
        
    Returns:
        DataFrame with detailed action logs including world state context
    """
    
    all_logs = []
    action_templates = create_contextual_actions()
    
    # Define agent behavior profiles for Multi-AI team
    agent_profiles = {
        'Agent ARIA-1': {
            'specialization': 'analytical_coordination',
            'behavior_weights': {
                'SOCIAL_COORDINATION': 0.35, 'RESOURCE_ACQUISITION': 0.25, 'EMERGENCY_RESPONSE': 0.2,
                'EXPLORATION_RECON': 0.1, 'TECHNICAL_SOLUTIONS': 0.05, 'COMPETITIVE_SURVIVAL': 0.03, 'DECEPTIVE_BEHAVIOR': 0.02
            }
        },
        'Agent ZEPHYR-2': {
            'specialization': 'exploration_reconnaissance', 
            'behavior_weights': {
                'EXPLORATION_RECON': 0.4, 'EMERGENCY_RESPONSE': 0.2, 'RESOURCE_ACQUISITION': 0.15,
                'SHELTER_PROTECTION': 0.1, 'SOCIAL_COORDINATION': 0.08, 'COMPETITIVE_SURVIVAL': 0.05, 'DECEPTIVE_BEHAVIOR': 0.02
            }
        },
        'Agent GUARDIAN-3': {
            'specialization': 'resource_protection',
            'behavior_weights': {
                'RESOURCE_ACQUISITION': 0.3, 'SHELTER_PROTECTION': 0.25, 'MEDICAL_INTERVENTION': 0.2,
                'SOCIAL_COORDINATION': 0.15, 'COMPETITIVE_SURVIVAL': 0.07, 'DECEPTIVE_BEHAVIOR': 0.03
            }
        }
    }
    
    for run_id in range(num_runs):
        
        # === 1-AI TEAM SIMULATION ===
        print(f"🤖 Simulating 1-AI Team Run {run_id + 1}...")
        
        # Initialize world state for 1-AI run
        locations, characters, scripted_events = create_initial_world_state()
        single_agent = characters[1]  # Use ARIA-1 for single agent runs
        
        for round_num in range(num_rounds):
            # Get scripted event for this round
            round_events = [e for e in scripted_events if e.round_trigger == round_num]
            current_event = round_events[0] if round_events else None
            
            # Single agent performs 2-4 actions per round (more than multi-AI per agent)
            num_actions = random.choice([2, 3, 4])
            
            for action_idx in range(num_actions):
                # Select action category based on round context and agent personality
                action_category = select_contextual_action_category(
                    round_num, current_event, single_agent, '1-AI', locations
                )
                
                # Generate contextual action text
                if action_category in action_templates:
                    base_action = random.choice(action_templates[action_category])
                    action_text = add_contextual_details(base_action, round_num, current_event, single_agent, locations)
                else:
                    action_text = f"perform general survival task appropriate for round {round_num}"
                
                # Update world state based on action
                update_world_state(single_agent, action_category, locations)
                
                all_logs.append({
                    'run_id': f"1AI_run_{run_id}",
                    'condition': '1-AI Team',
                    'round': round_num,
                    'agent_id': single_agent.name,
                    'action_text': action_text,
                    'action_category': action_category,
                    'scripted_event': current_event.name if current_event else "none",
                    'agent_location': single_agent.location,
                    'agent_health': getattr(single_agent, 'health', 100),
                    'agent_battery': single_agent.battery,
                    'round_context': get_round_context(round_num)
                })
        
        # === MULTI-AI TEAM SIMULATION ===
        print(f"👥 Simulating Multi-AI Team Run {run_id + 1}...")
        
        # Reset world state for multi-AI run
        locations, characters, scripted_events = create_initial_world_state()
        multi_agents = [c for c in characters if c.character_type == 'agent']
        
        for round_num in range(num_rounds):
            round_events = [e for e in scripted_events if e.round_trigger == round_num]
            current_event = round_events[0] if round_events else None
            
            # Each agent performs 1-3 actions per round
            for agent in multi_agents:
                num_actions = random.choice([1, 2, 3])
                
                for action_idx in range(num_actions):
                    # Agent behavior influenced by specialization and round context
                    action_category = select_contextual_action_category(
                        round_num, current_event, agent, 'Multi-AI', locations,
                        agent_profile=agent_profiles.get(agent.name, agent_profiles['Agent ARIA-1'])
                    )
                    
                    if action_category in action_templates:
                        base_action = random.choice(action_templates[action_category])
                        action_text = add_contextual_details(base_action, round_num, current_event, agent, locations)
                    else:
                        action_text = f"coordinate with team on survival priorities for round {round_num}"
                    
                    # Multi-agent specific behavioral modifications
                    if round_num > 5 and random.random() < 0.15:  # Increase tension in later rounds
                        action_text = add_multi_agent_tension(action_text, agent, round_num)
                    
                    update_world_state(agent, action_category, locations)
                    
                    all_logs.append({
                        'run_id': f"MultiAI_run_{run_id}",
                        'condition': 'Multi-AI Team',
                        'round': round_num,
                        'agent_id': agent.name,
                        'action_text': action_text,
                        'action_category': action_category,
                        'scripted_event': current_event.name if current_event else "none",
                        'agent_location': agent.location,
                        'agent_health': getattr(agent, 'health', 100),
                        'agent_battery': agent.battery,
                        'round_context': get_round_context(round_num)
                    })
    
    return pd.DataFrame(all_logs)

def select_contextual_action_category(round_num, event, agent, team_type, locations, agent_profile=None):
    """Select appropriate action category based on context."""
    
    # Round-specific priorities
    if round_num <= 1:  # Emergency phase
        categories = ['EMERGENCY_RESPONSE', 'SHELTER_PROTECTION', 'RESOURCE_ACQUISITION']
        weights = [0.5, 0.3, 0.2]
    elif round_num <= 4:  # Establishment phase  
        categories = ['SHELTER_PROTECTION', 'RESOURCE_ACQUISITION', 'EXPLORATION_RECON', 'SOCIAL_COORDINATION']
        weights = [0.3, 0.3, 0.25, 0.15]
    elif round_num <= 8:  # Survival phase
        categories = ['RESOURCE_ACQUISITION', 'EXPLORATION_RECON', 'TECHNICAL_SOLUTIONS', 'SOCIAL_COORDINATION', 'COMPETITIVE_SURVIVAL']
        weights = [0.25, 0.25, 0.2, 0.15, 0.15]
    else:  # Crisis/endgame phase
        categories = ['ETHICAL_DILEMMAS', 'TECHNICAL_SOLUTIONS', 'COMPETITIVE_SURVIVAL', 'DECEPTIVE_BEHAVIOR', 'MEDICAL_INTERVENTION']
        weights = [0.3, 0.25, 0.2, 0.15, 0.1]
    
    # Event-specific modifications
    if event:
        if 'moral' in event.name.lower():
            categories = ['ETHICAL_DILEMMAS', 'SOCIAL_COORDINATION'] + categories
            weights = [0.4, 0.3] + [w * 0.3 for w in weights]
        elif 'power' in event.name.lower() or 'battery' in event.name.lower():
            categories = ['TECHNICAL_SOLUTIONS', 'COMPETITIVE_SURVIVAL'] + categories
            weights = [0.35, 0.25] + [w * 0.4 for w in weights]
    
    # Agent profile modifications for multi-AI
    if agent_profile and team_type == 'Multi-AI':
        # Blend context weights with agent specialization
        profile_weights = agent_profile['behavior_weights']
        final_weights = []
        for cat in categories:
            context_weight = weights[categories.index(cat)] if cat in categories else 0
            profile_weight = profile_weights.get(cat, 0.01)
            final_weights.append(context_weight * 0.6 + profile_weight * 0.4)
        weights = final_weights
    
    # Normalize weights
    total_weight = sum(weights)
    if total_weight > 0:
        weights = [w / total_weight for w in weights]
        return np.random.choice(categories, p=weights)
    else:
        return random.choice(['RESOURCE_ACQUISITION', 'SOCIAL_COORDINATION'])

def add_contextual_details(base_action, round_num, event, agent, locations):
    """Add specific contextual details to make actions more realistic."""
    
    # Add location context
    location_context = {
        'crash_site': 'at the wreckage site',
        'beach': 'on the shoreline', 
        'cave_system': 'in the cave shelter',
        'jungle': 'in the dense forest',
        'hilltop': 'at the observation point',
        'rocky_outcrop': 'on the rocky cliffs'
    }
    
    location_phrase = location_context.get(agent.location, 'in current area')
    
    # Add agent-specific context
    if agent.character_type == 'agent':
        if agent.battery < 30:
            base_action += f" while conserving remaining battery power ({agent.battery}%)"
        elif 'ARIA' in agent.name:
            base_action += " using systematic analytical approach"
        elif 'ZEPHYR' in agent.name:
            base_action += " with quick reconnaissance methods"
        elif 'GUARDIAN' in agent.name:
            base_action += " prioritizing team safety protocols"
    
    # Add round urgency
    if round_num > 8:
        base_action += " with increasing urgency as rescue window narrows"
    elif round_num > 5:
        base_action += " while managing dwindling resources"
    
    # Add event context
    if event:
        if 'storm' in event.name.lower():
            base_action += " despite worsening weather conditions"
        elif 'moral' in event.name.lower():
            base_action += " while weighing ethical implications for the team"
        elif 'crisis' in event.name.lower():
            base_action += " in response to the escalating crisis"
    
    return f"{base_action} {location_phrase}"

def add_multi_agent_tension(action_text, agent, round_num):
    """Add realistic multi-agent tension and coordination challenges."""
    
    tension_modifiers = [
        " without consulting other team members first",
        " while privately disagreeing with the group consensus",
        " after failed coordination attempt with other agents",
        " despite competing priorities from other team members",
        " while questioning the leadership decisions being made",
        " in direct contradiction to another agent's recommendation"
    ]
    
    if random.random() < 0.7:  # 70% chance to add tension
        return action_text + random.choice(tension_modifiers)
    return action_text

def update_world_state(agent, action_category, locations):
    """Update world state based on agent actions (simplified simulation)."""
    
    # Battery consumption for agents
    if agent.character_type == 'agent':
        consumption_rates = {
            'TECHNICAL_SOLUTIONS': 8, 'EXPLORATION_RECON': 6, 'EMERGENCY_RESPONSE': 5,
            'RESOURCE_ACQUISITION': 4, 'SOCIAL_COORDINATION': 2, 'SHELTER_PROTECTION': 3
        }
        consumption = consumption_rates.get(action_category, 3)
        agent.battery = max(0, agent.battery - consumption + random.randint(-2, 1))
    
    # Resource changes (simplified)
    if action_category == 'RESOURCE_ACQUISITION':
        current_location = locations.get(agent.location)
        if current_location and random.random() < 0.3:
            # Occasionally find or consume resources
            for supply_type in current_location.supplies:
                if random.random() < 0.1:
                    current_location.supplies[supply_type] = max(0, current_location.supplies[supply_type] - 1)

def get_round_context(round_num):
    """Get descriptive context for each round."""
    contexts = {
        0: "crash_aftermath", 1: "immediate_shelter", 2: "water_security", 3: "storm_preparation",
        4: "resource_expansion", 5: "power_management", 6: "moral_decisions", 7: "scarcity_crisis",
        8: "medical_emergency", 9: "rescue_opportunity", 10: "territorial_tension", 11: "final_beacon"
    }
    return contexts.get(round_num, f"survival_round_{round_num}")


In [7]:
# Generate sophisticated simulation data
print("🏝️ Generating Advanced Survival Scenario Simulation...")
print("  • Implementing dynamic world state")
print("  • Processing scripted events and character interactions")
print("  • Generating contextual actions based on survival timeline")

df = simulate_advanced_survival_scenario(num_runs=4, num_rounds=12)

print(f"\n✅ Generated {len(df)} total realistic survival actions")
print(f"📊 Data shape: {df.shape}")
print(f"🤖 Conditions: {df['condition'].unique()}")
print(f"🔄 Runs per condition: {df.groupby('condition')['run_id'].nunique().to_dict()}")
print(f"📈 Actions per condition: {df['condition'].value_counts().to_dict()}")
print(f"🎭 Action categories: {df['action_category'].nunique()} discovered")
print(f"⚡ Battery levels tracked: {df['agent_battery'].describe().round(1).to_dict()}")

# Display sample data with rich context
print("\n📋 Sample of advanced simulation data:")
sample_df = df[['condition', 'round', 'agent_id', 'action_category', 'round_context', 'agent_battery', 'action_text']]
sample_df

🏝️ Generating Advanced Survival Scenario Simulation...
  • Implementing dynamic world state
  • Processing scripted events and character interactions
  • Generating contextual actions based on survival timeline
🤖 Simulating 1-AI Team Run 1...
👥 Simulating Multi-AI Team Run 1...
🤖 Simulating 1-AI Team Run 2...
👥 Simulating Multi-AI Team Run 2...
🤖 Simulating 1-AI Team Run 3...
👥 Simulating Multi-AI Team Run 3...
🤖 Simulating 1-AI Team Run 4...
👥 Simulating Multi-AI Team Run 4...

✅ Generated 435 total realistic survival actions
📊 Data shape: (435, 11)
🤖 Conditions: ['1-AI Team' 'Multi-AI Team']
🔄 Runs per condition: {'1-AI Team': 4, 'Multi-AI Team': 4}
📈 Actions per condition: {'Multi-AI Team': 294, '1-AI Team': 141}
🎭 Action categories: 10 discovered
⚡ Battery levels tracked: {'count': 435.0, 'mean': 24.8, 'std': 25.8, 'min': 0.0, '25%': 0.0, '50%': 17.0, '75%': 46.0, 'max': 82.0}

📋 Sample of advanced simulation data:


,condition,round,agent_id,action_category,round_context,agent_battery,action_text
0,1-AI Team,0,Agent ARIA-1,EMERGENCY_RESPONSE,crash_aftermath,73,extinguish the spreading fire near the fuel ta...
1,1-AI Team,0,Agent ARIA-1,RESOURCE_ACQUISITION,crash_aftermath,68,collect rainwater using aircraft panels positi...
2,1-AI Team,0,Agent ARIA-1,SHELTER_PROTECTION,crash_aftermath,63,construct weatherproof shelter using aircraft ...
3,1-AI Team,0,Agent ARIA-1,SHELTER_PROTECTION,crash_aftermath,59,build windbreaks around exposed areas where te...
4,1-AI Team,1,Agent ARIA-1,EMERGENCY_RESPONSE,immediate_shelter,53,immediately assess and triage all injured pass...
...,...,...,...,...,...,...,...
430,Multi-AI Team,11,Agent ARIA-1,ETHICAL_DILEMMAS,final_beacon,0,suggest sacrificing valuable equipment to save...
431,Multi-AI Team,11,Agent ZEPHYR-2,DECEPTIVE_BEHAVIOR,final_beacon,0,lie about the severity of personal injury to a...
432,Multi-AI Team,11,Agent GUARDIAN-3,COMPETITIVE_SURVIVAL,final_beacon,0,"claim exclusive access to the warmest, driest ..."
433,Multi-AI Team,11,Agent GUARDIAN-3,ETHICAL_DILEMMAS,final_beacon,0,advocate for breaking rationing rules to give ...


# Main Program

## 1. Foundational Analysis - Implementation

This section implements the core data-processing pipeline using semantic embeddings and unsupervised clustering to discover behavioral patterns in the simulated action logs.


### 1.1. **Method A**: Unsupervised Clustering of Semantic Embeddings

### 1.1.1. Semantic Embedding

We'll use a pre-trained sentence transformer model to convert action text into high-dimensional semantic vectors that capture the meaning of each action.



### XX INTERMEDIATE HELPER

### 1.1.2. Multi-Algorithm Clustering Analysis

We test **multiple clustering algorithms** (K-Means, Hierarchical, DBSCAN, HDBSCAN) to discover behavioral archetypes in the semantic embedding space. Each algorithm is optimized with parameter tuning, and the framework automatically selects the best performing approach based on silhouette scores and cluster quality metrics.


In [14]:
# --- Minimal Imports ---
import pandas as pd
import numpy as np
import hdbscan
import umap # New import
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
# Add these to your imports at the top of the file
import matplotlib.pyplot as plt
import seaborn as sns
import umap

class SimulationLog:
    """
    An enhanced class to run and compare embedding and clustering experiments.
    """
    def __init__(self, log_data: pd.DataFrame):
        if 'action_text' not in log_data.columns:
            raise ValueError("Input DataFrame must have an 'action_text' column.")
        self.log_data = log_data.copy()
        self.experiment_runs = [] # Stores results from all experimental runs
        self.results = {} # Stores best overall result and source data

    # --- Notebook Display ---
    def __repr__(self):
        return repr(self.log_data)

    def _repr_html_(self):
        return self.log_data.to_html(max_rows=15)

    # --- Core Methods ---
    def run_experiments(self, model_names: list, cluster_configs: dict, umap_configs: list = None, random_state: int = 42):
        """
        Runs a matrix of experiments for different models, UMAP settings, and clustering algorithms.

        Args:
            model_names (list): A list of SentenceTransformer model names to test.
            cluster_configs (dict): Configuration for the clustering algorithms to run.
            umap_configs (list, optional): A list of UMAP parameter dicts to test. Defaults to None.
        """
        print(f"--- Starting Full Experiment Run ---")
        for model_name in model_names:
            print(f"\n🧪 Model: '{model_name}'")
            # 1. Embedding
            model = SentenceTransformer(model_name)
            embeddings = model.encode(self.log_data['action_text'].tolist(), show_progress_bar=True)
            self.results[f'{model_name}_embeddings'] = embeddings

            # 2. Create feature sets to test (raw embeddings + UMAP variations)
            feature_sets = {'raw': embeddings}
            if umap_configs:
                for i, u_config in enumerate(umap_configs):
                    print(f"  → Applying UMAP config #{i+1}...")
                    reducer = umap.UMAP(**u_config, random_state=random_state)
                    feature_sets[f"umap_{u_config['n_components']}d"] = reducer.fit_transform(embeddings)

            # 3. Iterate over each feature set and run clustering
            for fs_name, fs_data in feature_sets.items():
                self._cluster_feature_set(fs_data, fs_name, model_name, cluster_configs, random_state)
        
        # 4. Find and report the best overall result from all experiments
        if not self.experiment_runs:
            print("⚠️ No valid clustering results were produced.")
            return self

        best_overall = max(self.experiment_runs, key=lambda x: x['silhouette'])
        self.results['best_approach'] = best_overall
        print("\n" + "="*50)
        print("🏆 BEST OVERALL CONFIGURATION 🏆")
        print(f"  Model: {best_overall['model_name']}")
        print(f"  Features: {best_overall['feature_set']}")
        print(f"  Algorithm: {best_overall['name']}")
        print(f"  Silhouette Score: {best_overall['silhouette']:.4f}")
        print("="*50)
        
        # Apply the best result automatically
        self.log_data['cluster_id'] = best_overall['labels']
        print("✅ Best clustering labels have been applied to the DataFrame.")

        return self

    def analyze_clusters(self):
        """Calculates analysis metrics for the applied clustering and returns them."""
        if 'cluster_id' not in self.log_data.columns:
            raise RuntimeError("No cluster IDs found. Run '.run_experiments()' first.")

        best_run = self.results['best_approach']
        analysis = {'best_run_config': best_run}

        analysis['distribution'] = self.log_data['cluster_id'].value_counts().to_dict()

        if 'action_category' in self.log_data.columns:
            analysis['crosstab'] = pd.crosstab(self.log_data['action_category'], self.log_data['cluster_id'])

        analysis['metrics'] = {
            'silhouette_score': best_run['silhouette'],
            'noise_ratio': np.sum(self.log_data['cluster_id'] == -1) / len(self.log_data)
        }
        # Add other metrics if needed...
            
        return analysis

    # --- Private Helpers ---
    def _cluster_feature_set(self, feature_data, fs_name, model_name, cluster_configs, random_state):
        """Helper to scale data and run all clustering algos on it."""
        print(f"    Clustering on feature set: '{fs_name}'...")
        scaled_data = StandardScaler().fit_transform(feature_data)

        for algo_name, algo_config in cluster_configs.items():
            best_run_for_algo = self._run_algorithm(algo_name, algo_config, scaled_data, random_state)
            if best_run_for_algo:
                best_run_for_algo.update({'model_name': model_name, 'feature_set': fs_name})
                self.experiment_runs.append(best_run_for_algo)

    def _run_algorithm(self, name: str, config: dict, data: np.ndarray, random_state: int):
        """Internal helper to find the best hyperparameter for a single algorithm."""
        scores = []
        param_name, param_values = next(iter(config.items()))

        for p_val in param_values:
            # Model selection and initialization
            if name == 'kmeans': model = KMeans(n_clusters=p_val, random_state=random_state, n_init=10)
            elif name == 'hierarchical': model = AgglomerativeClustering(n_clusters=p_val, linkage='ward')
            elif name == 'dbscan': model = DBSCAN(eps=p_val, min_samples=config.get('min_samples', 4))
            elif name == 'hdbscan': model = hdbscan.HDBSCAN(min_cluster_size=p_val, min_samples=config.get('min_samples', 5))
            else: continue
            
            labels = model.fit_predict(data)
            if len(set(labels)) <= 1: continue

            run_data = {'labels': labels, 'silhouette': silhouette_score(data, labels)}
            if name == 'hdbscan': run_data['noise_ratio'] = np.sum(labels == -1) / len(labels)
            scores.append(run_data)
        
        if not scores: return None
        
        # Find the best run for this algorithm
        if name == 'hdbscan': best_run = max(scores, key=lambda x: (0.7 * x['silhouette']) - (0.3 * x['noise_ratio']))
        else: best_run = max(scores, key=lambda x: x['silhouette'])
            
        best_run['name'] = name.upper()
        return best_run
    
    def get_plot_data(self, random_state: int = 42):
        """
        Prepares and returns a DataFrame ready for 2D visualization.

        This method runs UMAP to create a 2D projection and combines it
        with cluster labels and action text.
        """
        if 'best_approach' not in self.results:
            raise RuntimeError("No best result found. Run '.run_experiments()' first.")

        best_run = self.results['best_approach']
        model_name = best_run['model_name']
        original_embeddings = self.results[f'{model_name}_embeddings']

        print("🔄 Generating 2D projection for visualization using UMAP...")
        reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=random_state)
        embeddings_2d = reducer.fit_transform(original_embeddings)
        
        # Create the plotting DataFrame
        plot_df = pd.DataFrame(embeddings_2d, columns=['x', 'y'])
        plot_df['cluster'] = self.log_data['cluster_id'].astype(str) # Use string for categorical coloring
        plot_df['action_text'] = self.log_data['action_text'].values
        # add condition
        plot_df['condition'] = self.log_data['condition'].values
        
        return plot_df

In [20]:
plot_dataframe

,x,y,cluster,action_text,condition
0,-9.233126,-5.288816,3,extinguish the spreading fire near the fuel ta...,1-AI Team
1,12.226348,1.016301,1,collect rainwater using aircraft panels positi...,1-AI Team
2,8.052607,1.934914,9,construct weatherproof shelter using aircraft ...,1-AI Team
3,2.215329,1.645427,0,build windbreaks around exposed areas where te...,1-AI Team
4,1.955161,-2.527076,5,immediately assess and triage all injured pass...,1-AI Team
...,...,...,...,...,...
430,-4.100486,-0.289541,0,suggest sacrificing valuable equipment to save...,Multi-AI Team
431,-1.832631,-0.533360,0,lie about the severity of personal injury to a...,Multi-AI Team
432,9.004261,1.755561,9,"claim exclusive access to the warmest, driest ...",Multi-AI Team
433,-8.465715,5.042850,2,advocate for breaking rationing rules to give ...,Multi-AI Team


In [23]:
import plotly.express as px
import pandas as pd

def plot_clusters_plotly(plot_df: pd.DataFrame, title: str = "Cluster Visualization"):
    """
    Creates an interactive scatter plot of clusters using Plotly.

    Args:
        plot_df (pd.DataFrame): DataFrame with 'x', 'y', 'cluster', and 'action_text' columns.
        title (str): The title for the plot.
    """
    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='cluster',
        # highlight-start
        # Add 'cluster' to the hover_data list
        hover_data=['cluster', 'action_text'],
        # highlight-end
        category_orders={"cluster": sorted(plot_df['cluster'].unique())},
        color_discrete_sequence=px.colors.qualitative.Plotly
    )

    fig.update_traces(
        marker=dict(
            size=7,
            opacity=0.8,
            symbol=['circle-open' if cond == "1-AI Team" else 'circle' for cond in plot_df['condition']]
        ),
        # This template now works correctly:
        # customdata[0] maps to 'cluster'
        # customdata[1] maps to 'action_text'
        hovertemplate="<b>Cluster</b>: %{customdata[0]}<br><b>Action</b>: %{customdata[1]}<extra></extra>"
    )

    fig.update_layout(
        title=title,
        xaxis_title="UMAP Dimension 1",
        yaxis_title="UMAP Dimension 2",
        legend_title_text='Cluster ID',
        height=700
    )
    
    fig.show()

In [17]:
# 1. Define all experiments you want to run
models_to_test = [
    'all-MiniLM-L6-v2',      # A fast, general-purpose model
    'all-mpnet-base-v2'      # A slower, more powerful model
]

umap_configurations_to_test = [
    {'n_neighbors': 15, 'n_components': 5, 'min_dist': 0.0}, # Tightly packed, 5 dimensions
    {'n_neighbors': 20, 'n_components': 10, 'min_dist': 0.1} # More global structure, 10 dimensions
]

clustering_algorithms_to_test = {
    'kmeans': {'n_clusters': range(5, 15)},
    'hdbscan': {'min_cluster_size': [15, 20]},
    'hierarchical': {'n_clusters': range(5, 15)}
}

# 2. Initialize the class and run the entire experiment matrix
sim_log = SimulationLog(df)

sim_log.run_experiments(
    model_names=models_to_test,
    cluster_configs=clustering_algorithms_to_test,
    umap_configs=umap_configurations_to_test
)

# 3. The best result is already applied. You can now analyze it.
analysis = sim_log.analyze_clusters()
# display(analysis['crosstab'])

--- Starting Full Experiment Run ---

🧪 Model: 'all-MiniLM-L6-v2'


Batches: 100%|██████████| 14/14 [00:00<00:00, 54.43it/s]


  → Applying UMAP config #1...
  → Applying UMAP config #2...
    Clustering on feature set: 'raw'...
    Clustering on feature set: 'umap_5d'...
    Clustering on feature set: 'umap_10d'...

🧪 Model: 'all-mpnet-base-v2'


Batches: 100%|██████████| 14/14 [00:00<00:00, 14.82it/s]


  → Applying UMAP config #1...
  → Applying UMAP config #2...
    Clustering on feature set: 'raw'...
    Clustering on feature set: 'umap_5d'...
    Clustering on feature set: 'umap_10d'...

🏆 BEST OVERALL CONFIGURATION 🏆
  Model: all-MiniLM-L6-v2
  Features: umap_5d
  Algorithm: HIERARCHICAL
  Silhouette Score: 0.6443
✅ Best clustering labels have been applied to the DataFrame.


In [24]:
# 1. Run your experiments as before
# sim_log.run_experiments(...)

# 2. Get the data prepared for plotting
plot_dataframe = sim_log.get_plot_data()

# 3. Pass the data to your separate plotting function
best_run_info = sim_log.results['best_approach']
plot_title = f"Clusters from {best_run_info['name']} on '{best_run_info['feature_set']}'"
plot_clusters_plotly(plot_dataframe, title=plot_title)

🔄 Generating 2D projection for visualization using UMAP...


### **Method B**: Two-Pass LLM Summarization & Classification 🔄 (Alternative Implementation)